In [2]:
# import bioframe as bf
import torch
from gpzoo.kernels import batched_RBF, batched_MGGP_RBF
from torch import distributions
import matplotlib.pyplot as plt
from torch import nn, optim
from tqdm.autonotebook import tqdm
from gpzoo.utilities import add_jitter, whitened_KL
from gpzoo.gp import WSVGP, MGGP_WSVGP, GaussianPrior
import cooler
from matplotlib.colors import LogNorm
import numpy as np
import cooltools
import pandas as pd
import bioframe as bf
import os, subprocess

In [ ]:
class GPLVM(nn.Module):
    def __init__(self, prior, kernel, noise=0.1, Z_noise=0.1, jitter=1e-5):
        super().__init__()
        self.prior = prior
        self.kernel = kernel
        self.jitter = jitter
        self.noise = nn.Parameter(torch.log(torch.exp(torch.tensor(noise)) - 1))

    def forward(self, X, E=1, verbose=False, **kwargs):
        N = len(X)
        noise = torch.nn.functional.softplus(self.noise) #ensure positive

        qZ, pZ = self.prior()
        Z = qZ.rsample().T
        Z = torch.squeeze(Z)

        Kzz = kernel.forward(Z, Z)
        Kzz = Kzz.contiguous()
        Kzz = add_jitter(Kzz, self.jitter)
        Kzz.view(-1)[::N+1]+= (noise**2)

        pY = distributions.MultivariateNormal(torch.zeros_like(torch.squeeze(X)), Kzz)

        return pY, qZ, pZ